# LAVKA Prepare For Cube

Версия под новую структуру загрузки (`lower_snake_case`, историчность по `_source_file`/`_loaded_at`).


In [ ]:
from datetime import datetime, timedelta
import re

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, DoubleType, StringType

spark.conf.set("spark.databricks.io.cache.enabled", "true")
spark.conf.set("spark.databricks.photon.enabled", "true")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.localShuffleReader.enabled", "true")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 400 * 1024 * 1024)
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
spark.conf.set("spark.sql.shuffle.partitions", "auto")
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

TRGT_TABLE_METRICS = "ecom_sandbox.implant_cube_lavka"
TRGT_TABLE_METRICS_CITY = "ecom_sandbox.implant_cube_lavka_city"
TRGT_TABLE_ALERTS = "ecom_sandbox.implant_cube_lavka_alerts"
TRGT_TABLE_FORECAST_CHECK = "ecom_sandbox.implant_cube_lavka_forecast_check"

SOURCE_PO1 = "ecom_etl.lavka_po1"
SOURCE_MAPPING = "ecom_etl.lavka_po1_assort"

today = datetime.today().date()
cutoff_source_date = today - timedelta(days=120)
cutoff_output = cutoff_source_date
write_mode = "append"

print(f"cutoff_source_date={cutoff_source_date}, cutoff_output={cutoff_output}, write_mode={write_mode}")


In [ ]:
def normalize_col_name(name: str) -> str:
    s = re.sub(r"[^0-9a-zA-Z]+", "_", name.strip().lower())
    s = re.sub(r"_+", "_", s).strip("_")
    return s


def normalize_spark_columns(df):
    out = df
    for c in out.columns:
        nc = normalize_col_name(c)
        if nc != c:
            out = out.withColumnRenamed(c, nc)
    return out


def add_missing_cols(df, spec):
    out = df
    for col_name, spark_type in spec:
        if col_name not in out.columns:
            out = out.withColumn(col_name, F.lit(None).cast(spark_type))
    return out


def safe_numeric(col_name):
    return F.regexp_replace(F.col(col_name).cast("string"), ",", ".").cast(DoubleType())


def date_week_expr(col_name):
    return F.concat_ws("-", F.year(F.col(col_name)), F.format_string("%02d", F.weekofyear(F.col(col_name))))


In [ ]:
# Source load + schema standardization
df = normalize_spark_columns(spark.table(SOURCE_PO1))
mapping = normalize_spark_columns(spark.table(SOURCE_MAPPING))

required_po1 = [("date", "date"), ("city", "string"), ("supplier", "string"), ("item", "string"), ("item_name", "string"), ("metric", "string"), ("value", "string"), ("metric_date", "date")]
df = add_missing_cols(df, required_po1)

df = (
    df
    .withColumn("date", F.to_date(F.col("date")))
    .withColumn("metric_date", F.to_date(F.col("metric_date")))
    .withColumn("value_num", safe_numeric("value"))
)

df = df.filter(F.col("date").isNotNull() & F.col("metric").isNotNull() & F.col("item").isNotNull())
df = df.filter(F.col("date") >= F.lit(cutoff_source_date))

display(df.limit(20))


In [ ]:
# Dedup + FP horizons + weekly wide
ONHAND_METRICS = ["onhand_day_end_dc", "onhand_day_end_ds"]

dedup_window = Window.partitionBy("date", "city", "item", "metric").orderBy(F.desc("metric_date"), F.desc("_loaded_at"))

df_dedup = (
    df
    .filter(F.col("metric") != "order_document")
    .filter(F.col("metric") != "final_prediction")
    .withColumn("rn", F.row_number().over(dedup_window))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

all_metrics = sorted([r["metric"] for r in df_dedup.select("metric").distinct().collect()])
print(f"metrics_count={len(all_metrics)}")

df_fp_history = (
    df
    .filter(F.col("metric") == "final_prediction")
    .withColumn("weeks_before", (F.datediff(F.col("date"), F.col("metric_date")) / 7).cast(IntegerType()))
    .filter(F.col("weeks_before") >= 0)
    .withColumn("fp_col", F.concat(F.lit("final_prediction__w_minus_"), F.col("weeks_before").cast("string")))
)
fp_horizons = sorted([r["fp_col"] for r in df_fp_history.select("fp_col").distinct().collect()])
print(f"fp_horizons_count={len(fp_horizons)}")

df_dedup_weekly_base = df_dedup.withColumn(
    "week_monday",
    F.date_sub(F.col("date"), ((F.dayofweek(F.col("date")) + 5) % 7).cast(IntegerType()))
)

df_weekly_regular = (
    df_dedup_weekly_base
    .filter(~F.col("metric").isin(ONHAND_METRICS))
    .groupBy("week_monday", "city", "supplier", "item", "item_name", "metric")
    .agg(F.sum("value_num").alias("value_num"))
    .withColumnRenamed("week_monday", "date")
)

df_weekly_onhand = (
    df_dedup_weekly_base
    .filter(F.col("metric").isin(ONHAND_METRICS))
    .filter(F.col("date") == F.col("week_monday"))
    .groupBy("week_monday", "city", "supplier", "item", "item_name", "metric")
    .agg(F.first("value_num").alias("value_num"))
    .withColumnRenamed("week_monday", "date")
)

df_weekly_metrics = (
    df_weekly_regular.unionByName(df_weekly_onhand)
    .groupBy("date", "city", "supplier", "item", "item_name")
    .pivot("metric", all_metrics)
    .agg(F.first("value_num"))
)

df_weekly_fp = (
    df_fp_history
    .withColumn("forecast_week_monday", F.date_sub(F.col("metric_date"), ((F.dayofweek(F.col("metric_date")) + 5) % 7).cast(IntegerType())))
    .withColumn("date", F.date_sub(F.col("date"), ((F.dayofweek(F.col("date")) + 5) % 7).cast(IntegerType())))
    .withColumn("weeks_before", (F.datediff(F.col("date"), F.col("forecast_week_monday")) / 7).cast(IntegerType()))
    .filter(F.col("weeks_before") >= 0)
    .withColumn("fp_col", F.concat(F.lit("final_prediction__w_minus_"), F.col("weeks_before").cast("string")))
    .groupBy("date", "city", "supplier", "item", "item_name")
    .pivot("fp_col", fp_horizons)
    .agg(F.sum("value_num"))
)

df_weekly = df_weekly_metrics.join(df_weekly_fp, on=["date", "city", "supplier", "item", "item_name"], how="left")
display(df_weekly.limit(20))


In [ ]:
# Mapping + prices
mapping = add_missing_cols(mapping, [("lavka", "string"), ("gtin", "string"), ("sap_id", "string"), ("product_guid", "string"), ("vol_coeff", "double")])

df_weekly_mapped = (
    df_weekly
    .join(mapping.select("lavka", "gtin", "sap_id", "product_guid", "vol_coeff").dropDuplicates(["lavka"]), df_weekly["item"] == mapping["lavka"], "left")
    .drop("lavka")
)

start_date = df_weekly_mapped.select(F.min("date")).first()[0]

darkstore_city_to_state = {
    "Санкт-Петербург": "Санкт-Петербург", "Пушкин": "Санкт-Петербург", "Мурино": "Санкт-Петербург", "Кудрово": "Санкт-Петербург",
    "Колпино": "Санкт-Петербург", "Москва": "Москва", "Московская область": "Москва", "Домодедово": "Москва", "Балашиха": "Москва",
    "Раменское": "Москва", "Ивантеевка": "Москва", "Королёв": "Москва", "Подольск": "Москва", "Реутов": "Москва", "Химки": "Москва",
    "Щёлково": "Москва", "Видное": "Москва", "Лобня": "Москва", "Одинцово": "Москва", "Красногорск": "Москва", "Мытищи": "Москва",
    "Котельники": "Москва", "Люберцы": "Москва", "Новосибирск": "Новосибирск", "Казань": "Казань", "Иркутск": "Новосибирск",
    "Нижний Новгород": "Нижний Новгород", "Краснодар": "Краснодар", "Ростов-на-Дону": "Ростов-на-Дону", "Екатеринбург": "Екатеринбург",
    "Челябинск": "Челябинск", "Тюмень": "Тюмень", "Тула": "Тула", "Воронеж": "Воронеж", "Пермь": "Пермь", "Сочи": "Сочи",
    "Копейск": "Челябинск", "Яблоновский": "Краснодар"
}
mapping_df = spark.createDataFrame([(k, v) for k, v in darkstore_city_to_state.items()], ["darkstore_city", "city"])

brand = spark.table("ecom_etl.def_brand").filter(F.col("holding") == "PepsiCo")
product = (
    spark.table("ecom_etl.def_product")
    .join(F.broadcast(brand), on="brand_id", how="inner")
    .select("product_id", "barcode", "product_name", "brand_name")
)
darkstore = spark.table("ecom_etl.def_darkstore").filter(F.col("darkstore_group_platform") == "Яндекс.Лавка")
darkstore_mapped = darkstore.join(mapping_df, on="darkstore_city", how="left")

shelf = (
    spark.table("ecom_etl.def_e_shelf")
    .filter(F.col("date") >= F.lit(start_date))
    .join(product, on="product_id", how="inner")
    .join(F.broadcast(darkstore_mapped), on="darkstore_id", how="inner")
    .select("date", "barcode", "city", "customer_id", "darkstore_guid", "price_without_promo", "promo_price", "discount_percent")
)

weekly_shelf = (
    shelf
    .filter(F.dayofweek(F.col("date")) > 4)
    .withColumn("date", F.date_trunc("week", F.col("date")).cast("date"))
    .groupBy("date", "barcode", "city", "customer_id", "darkstore_guid")
    .agg(
        F.mode("price_without_promo").cast("double").alias("price_without_promo"),
        F.mode("promo_price").cast("double").alias("promo_price"),
        F.mode("discount_percent").cast("double").alias("discount_percent")
    )
    .withColumnRenamed("barcode", "gtin")
    .withColumnRenamed("darkstore_guid", "warehouse_guid")
)

joined_with_price = (
    df_weekly_mapped
    .join(weekly_shelf, on=["date", "gtin", "city"], how="left")
    .withColumn("_thursday", F.date_add(F.col("date"), 3))
    .withColumn("date_year", F.year("_thursday"))
    .withColumn("date_month", F.month("_thursday"))
    .withColumn("date_week", date_week_expr("date"))
    .withColumn("start_of_week", F.col("date"))
    .withColumn("city_nm", F.col("city"))
    .drop("_thursday")
)

display(joined_with_price.limit(20))


In [ ]:
# Final metrics and city metrics
final_metrics_df = (
    joined_with_price
    .withColumn("shortfalls_so", F.col("ordered_ds_so") - F.col("delivered_ds_so"))
    .withColumn("shortfalls_ds", F.col("ordered_ds_count") - F.col("delivered_ds_count"))
)

agg_cols = [
    "osa_plan", "osa_fact", "osa_loss", "ordered_ds_so", "delivered_ds_so", "ordered_ds_count", "delivered_ds_count",
    "sales_quantity", "sell_out_average", "remnants_ds", "remnants_ds_lag", "remnants_dc_per_ds_lag", "remnants_total",
    "2_forecast_ml", "3_forecast_ml", "4_forecast_ml", "2_forecast_cpfr",
    "ml_fa_numerator_lag_2", "ml_fa_numerator_lag_3", "ml_fa_numerator_lag_4", "cpfr_fa_numerator_lag_2"
]
agg_cols_present = [c for c in agg_cols if c in final_metrics_df.columns]

full_df_city = (
    final_metrics_df
    .groupBy("date_week", "date_year", "date_month", "start_of_week", "city_nm", "gtin", "product_guid")
    .agg(
        *[F.sum(c).alias(c) for c in agg_cols_present],
        F.mean("promo_price").alias("mean_promo_price"),
        F.mean("price_without_promo").alias("mean_price_without_promo"),
        F.mean("discount_percent").alias("mean_discount_percent"),
        F.avg("sales_quantity").alias("avg_sell_out_per_warehouse"),
        F.count(F.when(F.col("sales_quantity").isNotNull(), F.col("warehouse_guid"))).alias("warehouse_count"),
        F.sum(
            F.when(
                (F.col("sell_out_average") > 0) & ((F.col("remnants_total") / F.col("sell_out_average")) * 7 >= 3),
                1
            ).otherwise(0)
        ).alias("isa_fact")
    )
    .withColumn("shortfalls_ds", F.col("ordered_ds_count") - F.col("delivered_ds_count"))
    .withColumn("shortfalls_so", F.col("ordered_ds_so") - F.col("delivered_ds_so"))
)

sap_vol_mapping = mapping.select("product_guid", "vol_coeff").dropDuplicates(["product_guid"]) if "vol_coeff" in mapping.columns else mapping.select("product_guid").dropDuplicates(["product_guid"]).withColumn("vol_coeff", F.lit(1.0))

def apply_vol(df_in):
    return (
        df_in
        .join(sap_vol_mapping, "product_guid", "left")
        .withColumn("vol_coeff", F.coalesce(F.col("vol_coeff"), F.lit(1.0)))
        .withColumn("sales_quantity_vol", F.col("sales_quantity") * F.col("vol_coeff"))
        .withColumn("remnants_ds_lag_vol", F.col("remnants_ds_lag") * F.col("vol_coeff"))
        .withColumn("remnants_dc_per_ds_lag_vol", F.col("remnants_dc_per_ds_lag") * F.col("vol_coeff"))
        .withColumn("remnants_total_vol", F.col("remnants_total") * F.col("vol_coeff"))
        .drop("vol_coeff")
    )

final_metrics_df_mapped = apply_vol(final_metrics_df)
full_metrics_city_df_mapped = apply_vol(full_df_city)

display(final_metrics_df_mapped.limit(20))


In [ ]:
# Write full + city
spark.sql(f"DELETE FROM {TRGT_TABLE_METRICS} WHERE start_of_week >= date('{cutoff_output}')")
spark.sql(f"DELETE FROM {TRGT_TABLE_METRICS_CITY} WHERE start_of_week >= date('{cutoff_output}')")

full_spec = [
    ("date_week", StringType()), ("date_year", IntegerType()), ("date_month", IntegerType()), ("start_of_week", "date"),
    ("product_guid", StringType()), ("gtin", StringType()), ("sap_id", StringType()), ("warehouse_guid", StringType()),
    ("customer_id", StringType()), ("city_nm", StringType()),
    ("price_without_promo", DoubleType()), ("promo_price", DoubleType()), ("discount_percent", DoubleType()),
    ("osa_plan", DoubleType()), ("osa_fact", DoubleType()), ("osa_loss", DoubleType()),
    ("ordered_ds_so", DoubleType()), ("delivered_ds_so", DoubleType()), ("ordered_ds_count", DoubleType()), ("delivered_ds_count", DoubleType()),
    ("shortfalls_so", DoubleType()), ("shortfalls_ds", DoubleType()),
    ("sales_quantity", DoubleType()), ("sales_quantity_vol", DoubleType()),
    ("sell_out_average", DoubleType()), ("remnants_ds_lag", DoubleType()), ("remnants_ds_lag_vol", DoubleType()),
    ("remnants_dc_per_ds_lag", DoubleType()), ("remnants_dc_per_ds_lag_vol", DoubleType()),
    ("remnants_total", DoubleType()), ("remnants_total_vol", DoubleType()),
    ("2_forecast_ml", DoubleType()), ("3_forecast_ml", DoubleType()), ("4_forecast_ml", DoubleType()), ("2_forecast_cpfr", DoubleType()),
    ("ml_fa_numerator_lag_2", DoubleType()), ("ml_fa_numerator_lag_3", DoubleType()), ("ml_fa_numerator_lag_4", DoubleType()), ("cpfr_fa_numerator_lag_2", DoubleType())
]
full_spec = [(c, t) for c, t in full_spec]
full_df_write = add_missing_cols(final_metrics_df_mapped.filter(F.col("start_of_week") >= F.lit(cutoff_source_date)), full_spec)
full_cols = [c for c, _ in full_spec]
(
    full_df_write
    .select(*full_cols)
    .filter(F.col("product_guid").isNotNull())
    .dropDuplicates(["date_week", "product_guid", "warehouse_guid"])
    .write.mode(write_mode).option("mergeSchema", "true").saveAsTable(TRGT_TABLE_METRICS)
)

city_spec = [
    ("date_week", StringType()), ("date_year", IntegerType()), ("date_month", IntegerType()), ("start_of_week", "date"),
    ("product_guid", StringType()), ("gtin", StringType()), ("sap_id", StringType()), ("city_nm", StringType()),
    ("warehouse_count", IntegerType()),
    ("mean_price_without_promo", DoubleType()), ("mean_promo_price", DoubleType()), ("mean_discount_percent", DoubleType()),
    ("osa_plan", DoubleType()), ("osa_fact", DoubleType()), ("osa_loss", DoubleType()), ("isa_fact", DoubleType()),
    ("ordered_ds_so", DoubleType()), ("delivered_ds_so", DoubleType()), ("ordered_ds_count", DoubleType()), ("delivered_ds_count", DoubleType()),
    ("shortfalls_so", DoubleType()), ("shortfalls_ds", DoubleType()),
    ("sales_quantity", DoubleType()), ("sales_quantity_vol", DoubleType()),
    ("sell_out_average", DoubleType()), ("remnants_ds_lag", DoubleType()), ("remnants_ds_lag_vol", DoubleType()),
    ("remnants_dc_per_ds_lag", DoubleType()), ("remnants_dc_per_ds_lag_vol", DoubleType()),
    ("remnants_total", DoubleType()), ("remnants_total_vol", DoubleType()),
    ("2_forecast_ml", DoubleType()), ("3_forecast_ml", DoubleType()), ("4_forecast_ml", DoubleType()), ("2_forecast_cpfr", DoubleType()),
    ("ml_fa_numerator_lag_2", DoubleType()), ("ml_fa_numerator_lag_3", DoubleType()), ("ml_fa_numerator_lag_4", DoubleType()), ("cpfr_fa_numerator_lag_2", DoubleType())
]
city_df_write = add_missing_cols(full_metrics_city_df_mapped.filter(F.col("start_of_week") >= F.lit(cutoff_output)), city_spec)
city_cols = [c for c, _ in city_spec]
(
    city_df_write
    .select(*city_cols)
    .filter(F.col("gtin").isNotNull())
    .write.mode(write_mode).option("mergeSchema", "true").saveAsTable(TRGT_TABLE_METRICS_CITY)
)

print("FULL and CITY metrics saved")


In [ ]:
# Alerts + forecast check
base_for_alerts = final_metrics_df_mapped.filter(F.col("remnants_ds_lag").isNotNull())

last_5_weeks = [r["date_week"] for r in base_for_alerts.select("date_week").distinct().orderBy(F.col("date_week").desc()).limit(5).collect()]
last_2_weeks = last_5_weeks[:2]

if len(last_2_weeks) == 2:
    w0, w1 = last_2_weeks[0], last_2_weeks[1]
    base_df = (
        base_for_alerts
        .filter(F.col("date_week").isin(last_5_weeks))
        .withColumn("osa", F.try_divide(F.col("osa_fact"), F.col("osa_plan")))
        .withColumn("sl", F.try_divide(F.col("delivered_ds_so"), F.col("ordered_ds_so")))
        .withColumn("dos", 7 * F.try_divide(F.col("remnants_total"), F.col("sell_out_average")))
        .withColumn("is_alert", (F.col("osa") < 0.95) & ((F.col("sl") > 0.8) | F.col("sl").isNull()) & (F.col("dos") < 6))
    )

    alert_pairs = (
        base_df
        .filter(F.col("date_week").isin(last_2_weeks))
        .groupBy("warehouse_guid", "product_guid")
        .agg(
            F.max(F.when((F.col("date_week") == w0) & F.col("is_alert"), 1).otherwise(0)).alias("alert_w0"),
            F.max(F.when((F.col("date_week") == w1) & F.col("is_alert"), 1).otherwise(0)).alias("alert_w1")
        )
        .filter((F.col("alert_w0") == 1) | (F.col("alert_w1") == 1))
        .select("warehouse_guid", "product_guid")
    )

    alerting_df = (
        base_df
        .join(alert_pairs, ["warehouse_guid", "product_guid"], "inner")
        .select("date_week", "warehouse_guid", "product_guid", "osa_plan", "osa_fact", "ordered_ds_so", "osa", "sl", "dos", "sales_quantity", "remnants_total")
        .orderBy("warehouse_guid", "product_guid", "date_week")
    )
    alerting_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TRGT_TABLE_ALERTS)
    print("alerts saved")
else:
    print("alerts skipped: not enough weeks")

full_df_fc = (
    final_metrics_df_mapped
    .join(spark.table("ecom_etl.td_product").select("gtin", "category").distinct(), on="gtin", how="inner")
)
sell_out_df = full_df_fc.dropna(subset=["sales_quantity"])
sell_out_weeks = [r["date_week"] for r in sell_out_df.select("date_week").distinct().orderBy(F.desc("date_week")).collect()]

if len(sell_out_weeks) >= 10:
    cutoff_rank = sell_out_weeks[5]
    cutoff_select = sell_out_weeks[9]

    top_sellers = (
        sell_out_df
        .filter(F.col("date_week") >= F.lit(cutoff_rank))
        .groupBy("category", "gtin")
        .agg(F.sum("sales_quantity").alias("total_sales"))
        .withColumn("rnk", F.row_number().over(Window.partitionBy("category").orderBy(F.desc("total_sales"))))
        .filter(F.col("rnk") <= 8)
        .select("gtin")
    )

    result_df = (
        full_df_fc
        .filter(F.col("date_week") >= F.lit(cutoff_select))
        .join(top_sellers, on=["gtin"], how="inner")
        .withColumn("osa", F.try_divide(F.col("osa_fact"), F.col("osa_plan")))
        .withColumn("sl", F.try_divide(F.col("delivered_ds_so"), F.col("ordered_ds_so")))
        .withColumn("dos", 7 * F.try_divide(F.col("remnants_total"), F.col("sell_out_average")))
        .select("date_week", "warehouse_guid", "product_guid", "gtin", "osa_plan", "osa_fact", "ordered_ds_so", "3_forecast_ml", "2_forecast_cpfr", "osa", "sl", "dos", "sales_quantity", "remnants_total")
        .orderBy("warehouse_guid", "gtin", "date_week")
    )

    result_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TRGT_TABLE_FORECAST_CHECK)
    print("forecast check saved")
else:
    print("forecast check skipped: not enough sell-out weeks")


In [ ]:
dbutils.notebook.exit("success")
